In [1]:
import pandas as pd
import scraping_functions as sf
import importlib

importlib.reload(sf);

In [2]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## Saving Data into CSV

Collecting data from Transfermarkt requires sending a large number of requests across multiple seasons and rounds, which can be time-consuming. To avoid repeating the scraping process every time the data is needed, the extracted information is saved as CSV files. For this project, the historical database covers all Premier League seasons from 1992 to 2025 (the latest completed season at the time of development).

### Iterating Through Functions That Only Require Seasons (Faster)

Some scraping functions only require the season as input, such as the final league table, champions, and squad information. These functions can be executed with a single loop that iterates through all seasons and stores the results in CSV files.

In [9]:
premier_table = []
premier_squad = []
premier_top_scorers = []
league = 'premier-league'


for season in range(1992, 2026):
    # Creating the table list
    table_season_data = sf.get_table(headers, league, season)
    premier_table.extend(table_season_data[1:])

    # Creating the squad List
    squad_season_data = sf.get_squad(headers,league,season)
    premier_squad.extend(squad_season_data[1:])

    # Creating the top scorers List
    table_data = sf.get_top_scorers(headers,league,season)
    premier_top_scorers.extend(table_data[1:])

# Transforming to a pandas Data Frame
df_premier_table = pd.DataFrame(premier_table, columns=table_season_data[0])
df_premier_squad = pd.DataFrame(premier_squad, columns=squad_season_data[0])
df_premier_top_scorers = pd.DataFrame(premier_top_scorers,columns=table_data[0])
df_premier_top_scorers.sort_values(['season_id','pos'],inplace=True,ignore_index=True)

# Exporting to csv
df_premier_table.to_csv('../data-csv/premier-table.csv', index=False)
df_premier_squad.to_csv('../data-csv/premier-squad.csv', index=False)
df_premier_top_scorers.to_csv('../data-csv/premier-top-scorers.csv', index=False)

### Iterating Through Functions That Require Seasons and Rounds

Other functions, such as match results, match events, and round-by-round standings, require both the season and the round as input. These functions are executed using nested loops, iterating through every round of every season before saving the collected data.

In [6]:
premier_events = []
premier_matches = []
premier_placements = []
league = 'premier-league'

for season in range(2024,2026):
    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        # Creating the events List
        event_season_data = sf.get_events(headers,league,season,n_round)
        premier_events.extend(event_season_data[1:])

        # Creating the matches List
        matches_data = sf.get_match(headers,league,season,n_round)
        premier_matches.extend(matches_data[1:])

        # Creating the placements List
        placements_data = sf.get_placements(headers,league,season,n_round)
        premier_placements.extend(placements_data[1:])

df_premier_events = pd.DataFrame(premier_events,columns=event_season_data[0])
df_premier_matches = pd.DataFrame(premier_matches, columns=matches_data[0])
df_premier_placements = pd.DataFrame(premier_placements, columns=placements_data[0])

df_premier_events.to_csv("../data-csv/premier-events.csv", index=False)
df_premier_matches.to_csv("../data-csv/premier-matches.csv", index=False)
df_premier_placements.to_csv("../data-csv/premier-placements.csv", index=False)


### Updating the Database with Future Seasons

Once the initial historical database has been created, future seasons can be added without rebuilding the entire dataset. The same functions can simply be executed for the new season, and the results appended to the existing CSV files, making the project scalable, reusable, and easy to maintain.

In [3]:
def unique_season_round(pandas_column,index_season,index_round):
    output = []

    for item in pandas_column:
        split_list = item.split('-')
        season = split_list[index_season]
        round = split_list[index_round]

        temp_tuple = (int(season),int(round))
        output.append(temp_tuple)

    output = set(output)    
    return output

In [31]:
test = pd.read_csv('../data-csv/premier-events.csv', encoding='UTF-8')

t_unique = unique_season_round(test['match_id'],1,2)
print(len(t_unique))

count = 0

for season in range(2022,2026):
    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        check = (season,n_round)
        if check in t_unique: break
        else: count += 1

print(f'urls scrapped = {count}')

display(test)

76
urls scrapped = 76


,season_id,match_id,event_id,event_team,event_minute,event_type,event_player
0,GB1-2024,M-2024-01-01,E-2024-01-0001,Manchester United,87',1,Joshua Zirkzee
1,GB1-2024,M-2024-01-02,E-2024-01-0002,Liverpool FC,60',1,Diogo Jota
2,GB1-2024,M-2024-01-02,E-2024-01-0003,Liverpool FC,65',1,Mohamed Salah
3,GB1-2024,M-2024-01-03,E-2024-01-0004,Arsenal FC,25',1,Kai Havertz
4,GB1-2024,M-2024-01-03,E-2024-01-0005,Arsenal FC,74',1,Bukayo Saka
...,...,...,...,...,...,...,...
2280,GB1-2025,M-2025-38-08,E-2025-38-0021,Chelsea FC,62',-3,Wesley Fofana
2281,GB1-2025,M-2025-38-09,E-2025-38-0022,Tottenham Hotspur,43',1,João Palhinha
2282,GB1-2025,M-2025-38-10,E-2025-38-0023,West Ham United,67',1,Taty Castellanos
2283,GB1-2025,M-2025-38-10,E-2025-38-0024,West Ham United,79',1,Jarrod Bowen


In [19]:
league = 'premier-league'

event_extended = pd.read_csv('../data-csv/premier-events.csv', encoding='UTF-8')
event_unique = unique_season_round(event_extended['match_id'],1,2)

for season in range(1992,2026):

    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        check = (season,n_round)
        if check in event_unique: break
        else:
            data = sf.get_events(headers,league,season,n_round)
            temp = pd.DataFrame(data[1:],columns=data[0])
            event_extended = pd.concat([event_extended,temp])    

event_extended.sort_values(['event_id'],ignore_index=True,inplace=True)

event_extended.to_csv("../data-csv/premier-events.csv", index=False)
display(event_extended)

,season_id,match_id,event_id,event_team,event_minute,event_type,event_player
0,GB1-1992,M-1992-01-01,E-1992-01-0001,Chelsea FC,84',1,Mick Harford
1,GB1-1992,M-1992-01-01,E-1992-01-0002,Oldham Athletic,86',1,Nick Henry
2,GB1-1992,M-1992-01-02,E-1992-01-0003,Coventry City,9',1,John Williams
3,GB1-1992,M-1992-01-02,E-1992-01-0004,Coventry City,51',1,David Smith
4,GB1-1992,M-1992-01-02,E-1992-01-0005,Middlesbrough FC,63',1,Paul Wilkinson
...,...,...,...,...,...,...,...
37871,GB1-2025,M-2025-38-08,E-2025-38-0021,Chelsea FC,62',-3,Wesley Fofana
37872,GB1-2025,M-2025-38-09,E-2025-38-0022,Tottenham Hotspur,43',1,João Palhinha
37873,GB1-2025,M-2025-38-10,E-2025-38-0023,West Ham United,67',1,Taty Castellanos
37874,GB1-2025,M-2025-38-10,E-2025-38-0024,West Ham United,79',1,Jarrod Bowen


In [33]:
league = 'premier-league'

matches_extended = pd.read_csv('../data-csv/premier-matches.csv', encoding='UTF-8')
matches_unique = unique_season_round(matches_extended['match_id'],1,2)

for season in range(2022,2026):

    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        check = (season,n_round)
        if check in matches_unique: break
        else:
            data = sf.get_match(headers,league,season,n_round)
            temp = pd.DataFrame(data[1:],columns=data[0])
            matches_extended = pd.concat([matches_extended,temp])    

matches_extended.sort_values(['match_id'],ignore_index=True,inplace=True)

display(matches_extended)

,season_id,round_id,match_id,home_team,final_score,away_team,date,referee,attendance
0,GB1-2022,R-2022-01,M-2022-01-001,Crystal Palace,0:2,Arsenal FC,05/08/2022,Anthony Taylor,25.286
1,GB1-2022,R-2022-01,M-2022-01-002,Fulham FC,2:2,Liverpool FC,06/08/2022,Andrew Madley,22.207
2,GB1-2022,R-2022-01,M-2022-01-003,Tottenham Hotspur,4:1,Southampton FC,06/08/2022,Andre Marriner,61.732
3,GB1-2022,R-2022-01,M-2022-01-004,Newcastle United,2:0,Nottingham Forest,06/08/2022,Simon Hooper,52.245
4,GB1-2022,R-2022-01,M-2022-01-005,Leeds United,2:1,Wolverhampton Wanderers,06/08/2022,Robert Jones,36.347
...,...,...,...,...,...,...,...,...,...
1505,GB1-2025,R-2025-38,M-2025-38-006,Manchester City,1:2,Aston Villa,24/05/2026,Andrew Madley,60.332
1506,GB1-2025,R-2025-38,M-2025-38-007,Nottingham Forest,1:1,AFC Bournemouth,24/05/2026,Craig Pawson,30.741
1507,GB1-2025,R-2025-38,M-2025-38-008,Sunderland AFC,2:1,Chelsea FC,24/05/2026,Chris Kavanagh,47.155
1508,GB1-2025,R-2025-38,M-2025-38-009,Tottenham Hotspur,1:0,Everton FC,24/05/2026,Michael Oliver,61.812


In [34]:
league = 'premier-league'

placements_extended = pd.read_csv('../data-csv/premier-placements.csv', encoding='UTF-8')
placements_unique = unique_season_round(placements_extended['round_id'],1,2)

for season in range(2022,2026):

    if season > 1994: season_round = 38
    else: season_round = 42

    for n_round in range(1,season_round+1):
        check = (season,n_round)
        if check in placements_unique: break
        else:
            data = sf.get_placements(headers,league,season,n_round)
            temp = pd.DataFrame(data[1:],columns=data[0])
            placements_extended = pd.concat([placements_extended,temp])    

placements_extended.sort_values(['round_id'],ignore_index=True,inplace=True)

display(placements_extended)

,season_id,round_id,placement,team_name,matches,wins,draws,losses,goals,goal_dif,points
0,GB1-2022,R-2022-01,20,Southampton FC,1,0,0,1,1:4,-3,0
1,GB1-2022,R-2022-01,19,West Ham United,1,0,0,1,0:2,-2,0
2,GB1-2022,R-2022-01,18,Nottingham Forest,1,0,0,1,0:2,-2,0
3,GB1-2022,R-2022-01,17,Crystal Palace,1,0,0,1,0:2,-2,0
4,GB1-2022,R-2022-01,16,Aston Villa,1,0,0,1,0:2,-2,0
...,...,...,...,...,...,...,...,...,...,...,...
3035,GB1-2025,R-2025-38,4,Aston Villa,38,19,8,11,56:49,7,65
3036,GB1-2025,R-2025-38,3,Manchester United,38,20,11,7,69:50,19,71
3037,GB1-2025,R-2025-38,2,Manchester City,38,23,9,6,77:35,42,78
3038,GB1-2025,R-2025-38,9,Brentford FC,38,14,11,13,55:52,3,53
